# 📊 实验二：数据探索与预处理

## 学习目标
- 理解多轮对话数据格式
- 将对话数据格式化为训练用的文本
- 构建训练/验证集

## 1. 加载数据集

项目提供的数据：
- `medical_multi_data.json` — 1596 条

In [ ]:
import json
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": "src/medical_multi_data.json"},
    split="train"
)
print(f"数据集大小: {len(dataset)} 条")

## 2. 探索数据结构

每条数据包含 `conversation` 字段，里面是多轮对话数组。

In [ ]:
sample = dataset[0]
print("字段:", list(sample.keys()))
print("对话轮数:", len(sample["conversation"]))
print("\n第一轮:")
print(json.dumps(sample["conversation"][0], indent=2, ensure_ascii=False))

In [ ]:
# 统计对话轮数分布
from collections import Counter

turn_counts = Counter()
for item in dataset:
    turn_counts[len(item["conversation"])] += 1

print("对话轮数分布:")
for turns, count in sorted(turn_counts.items()):
    print(f"  {turns} 轮: {count} 条 ({count/len(dataset)*100:.1f}%)")

## 3. 格式化数据

将 `conversation` 转换为训练用的文本格式。

In [ ]:
def convert_conversation(conversation):
    """将多轮对话转换为训练文本格式"""
    text = ""
    for turn in conversation:
        if turn.get("system"):
            text += f"<|system|>\n{turn['system']}\n<|end|>\n"
        text += f"<|user|>\n{turn['input']}\n<|end|>\n"
        text += f"<|assistant|>\n{turn['output']}\n<|end|>\n"
    return text.strip()

# 测试
sample_text = convert_conversation(dataset[0]["conversation"])
print(sample_text[:500] + "\n...")

## 4. 批量处理数据集

In [ ]:
def format_data(example):
    """批量格式化函数"""
    text = convert_conversation(example["conversation"])
    return {"text": text}

formatted_dataset = dataset.map(format_data, remove_columns=dataset.column_names)
print(f"格式化完成: {len(formatted_dataset)} 条")
print(formatted_dataset[0]["text"][:200] + "...")

## 5. 切分数据集

In [ ]:
split_data = formatted_dataset.train_test_split(test_size=0.1)
train_data = split_data["train"]
eval_data = split_data["test"]

print(f"训练集: {len(train_data)} 条")
print(f"验证集: {len(eval_data)} 条")

## 小结

✅ 理解了多轮对话数据结构
✅ 完成了数据格式化
✅ 构建了训练集和验证集

下一节将进行 LoRA 微调训练。

## 课后练习

1. (单选题) dataset.map(func, remove_columns=dataset.column_names) 执行后，数据集字段为？
   - A. 仅 func 返回的字段
   - B. 保留原字段
   - C. 原字段与返回字段共存
   - D. 空

2. (单选题) 对话数据某个 turn 缺少 system 字段，最合理的处理是？
   - A. 跳过 system 拼接，保留 user/assistant
   - B. 终止整个样本
   - C. 用空字符串占位
   - D. 随机生成 system

3. (多选题) SFT 文本模板中应包含？
   - A. system 指令
   - B. user 提问
   - C. assistant 回答
   - D. 结束符/分隔符

4. (多选题) train_test_split(test_size=0.1, seed=42) 的正确理解包括？
   - A. 同一 seed 可复现切分
   - B. 训练/验证样本不重叠
   - C. 能消除类别不平衡
   - D. 样本顺序被确定

5. (判断题) datasets 的 Dataset.map 默认按样本逐一处理，batched 参数默认为 False。

6. (判断题) SFT 训练样本以 user 提问结尾，不包含 assistant 回答。

7. (填空题) 格式化对话时，角色开始标记通常为 ____、____、____。

8. (填空题) 为保证 train/val 切分可复现，train_test_split 应设置 ____ 参数。

9. (简答题) 为什么拼接后的 text 需要 strip() 并在末尾添加结束标记？

10. (简答题) 如何检测训练/验证集之间的样本泄漏？

11. (代码设计题) 编写 prepare_dataset(json_path, seed=42)，完成 JSON 读取、对话格式化、9:1 切分并返回 train/val。

12. (单选题) 验证 loss 明显高于训练 loss 且差距随训练扩大，最可能是？
   - A. 过拟合
   - B. 欠拟合
   - C. 学习率过低
   - D. 数据全部重复

13. (多选题) 数据质量检查应覆盖？
   - A. 空文本/缺失角色
   - B. 长度异常
   - C. 训练验证重复样本
   - D. 标签/角色顺序错误

14. (判断题) 多轮对话样本应保留从 system 到最终 assistant 的完整轮次。

15. (简答题) 数据集仅 1596 条时，为什么 9:1 切分的验证集波动大？如何缓解？

> 参考答案见 answer/05.03_data_exploration_answer.ipynb。